In [1]:
!pip install selenium webdriver-manager pillow tqdm requests


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import time
import requests
from PIL import Image
from io import BytesIO
from tqdm import tqdm

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

In [3]:
BASE_DIR = "01_datasets"

DATASETS = {
    "facades": "building facade architecture",
    "streets": "urban street city architecture",
    "textures": "brick concrete wall texture"
}

TARGET_COUNT = 50

In [4]:
def create_folder(path):
    os.makedirs(path, exist_ok=True)


def download_image(url, save_path, size=(512, 512)):
    try:
        headers = {
            "User-Agent": "Mozilla/5.0"
        }

        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()

        img = Image.open(BytesIO(response.content)).convert("RGB")
        img.thumbnail(size)

        canvas = Image.new("RGB", size, (255, 255, 255))
        x = (size[0] - img.width) // 2
        y = (size[1] - img.height) // 2
        canvas.paste(img, (x, y))

        canvas.save(save_path, "JPEG", quality=90)
        return True

    except Exception as e:
        return False

In [5]:
def scrape_bing_image_urls(keyword, max_urls=50):
    chrome_options = Options()
    chrome_options.add_argument("--disable-search-engine-choice-screen")
    chrome_options.add_argument("--start-maximized")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=chrome_options
    )

    search_url = f"https://www.bing.com/images/search?q={keyword.replace(' ', '+')}"
    driver.get(search_url)

    time.sleep(5)

    image_urls = set()
    scroll_round = 0

    while len(image_urls) < max_urls and scroll_round < 15:
        image_elements = driver.find_elements(By.CSS_SELECTOR, "img.mimg")

        for img in image_elements:
            src = img.get_attribute("src")
            data_src = img.get_attribute("data-src")

            image_url = src or data_src

            if image_url and image_url.startswith("http"):
                image_urls.add(image_url)

            if len(image_urls) >= max_urls:
                break

        driver.find_element(By.TAG_NAME, "body").send_keys(Keys.END)
        time.sleep(2)
        scroll_round += 1

        print(f"{keyword}: collected {len(image_urls)} image URLs")

    driver.quit()

    return list(image_urls)

In [6]:
create_folder(BASE_DIR)

for dataset_name, keyword in DATASETS.items():
    folder = os.path.join(BASE_DIR, dataset_name)
    create_folder(folder)

    print(f"\n===== Scraping {dataset_name}: {keyword} =====")

    urls = scrape_bing_image_urls(keyword, max_urls=TARGET_COUNT * 2)

    print(f"Found {len(urls)} urls for {dataset_name}")

    count = 0

    for url in tqdm(urls):
        if count >= TARGET_COUNT:
            break

        filename = f"{dataset_name}_{count + 1:03d}.jpg"
        save_path = os.path.join(folder, filename)

        success = download_image(url, save_path)

        if success:
            count += 1

        time.sleep(0.3)

    print(f"Downloaded {count} images into {folder}")


===== Scraping facades: building facade architecture =====
building facade architecture: collected 25 image URLs
building facade architecture: collected 60 image URLs
building facade architecture: collected 95 image URLs
building facade architecture: collected 100 image URLs
Found 100 urls for facades


 50%|████████████████████████████████████████▌                                        | 50/100 [00:43<00:43,  1.15it/s]


Downloaded 50 images into 01_datasets\facades

===== Scraping streets: urban street city architecture =====
urban street city architecture: collected 60 image URLs
urban street city architecture: collected 95 image URLs
urban street city architecture: collected 100 image URLs
Found 100 urls for streets


 50%|████████████████████████████████████████▌                                        | 50/100 [00:40<00:40,  1.25it/s]


Downloaded 50 images into 01_datasets\streets

===== Scraping textures: brick concrete wall texture =====
brick concrete wall texture: collected 28 image URLs
brick concrete wall texture: collected 98 image URLs
brick concrete wall texture: collected 100 image URLs
Found 100 urls for textures


 50%|████████████████████████████████████████▌                                        | 50/100 [00:41<00:41,  1.20it/s]

Downloaded 50 images into 01_datasets\textures


In [7]:
for folder in ["facades", "streets", "textures"]:
    path = os.path.join("01_datasets", folder)
    files = [
        f for f in os.listdir(path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ] if os.path.exists(path) else []

    print(folder, len(files))

facades 53
streets 52
textures 53


In [8]:
import os
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from tqdm import tqdm

BASE_DIR = "01_datasets"

DATASETS = {
    "facades": [
        "building facade architecture",
        "modern building facade",
        "architectural facade pattern",
        "urban building elevation",
        "contemporary facade design",
        "building exterior architecture"
    ],
    "streets": [
        "urban street city architecture",
        "city street buildings",
        "street view architecture",
        "urban landscape street",
        "city road buildings",
        "downtown street photography"
    ],
    "textures": [
        "brick wall texture",
        "concrete wall texture",
        "wood grain texture",
        "stone material texture",
        "weathered wall texture",
        "architectural material texture"
    ]
}

TARGET_COUNT = 200

In [9]:
def scrape_bing_image_urls(keyword, max_urls=80):
    chrome_options = Options()
    chrome_options.add_argument("--disable-search-engine-choice-screen")
    chrome_options.add_argument("--start-maximized")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=chrome_options
    )

    search_url = f"https://www.bing.com/images/search?q={keyword.replace(' ', '+')}"
    driver.get(search_url)

    time.sleep(4)

    image_urls = set()
    scroll_round = 0

    while len(image_urls) < max_urls and scroll_round < 20:
        image_elements = driver.find_elements(By.CSS_SELECTOR, "img.mimg")

        for img in image_elements:
            src = img.get_attribute("src")
            data_src = img.get_attribute("data-src")
            image_url = src or data_src

            if image_url and image_url.startswith("http"):
                image_urls.add(image_url)

            if len(image_urls) >= max_urls:
                break

        driver.find_element(By.TAG_NAME, "body").send_keys(Keys.END)
        time.sleep(1.5)
        scroll_round += 1

        print(f"{keyword}: collected {len(image_urls)} urls")

    driver.quit()
    return list(image_urls)

In [10]:
for dataset_name, keywords in DATASETS.items():
    folder = os.path.join(BASE_DIR, dataset_name)
    os.makedirs(folder, exist_ok=True)

    existing_files = [
        f for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    count = len(existing_files)
    print(f"\n===== {dataset_name}: currently {count} images =====")

    for keyword in keywords:
        if count >= TARGET_COUNT:
            break

        print(f"\nScraping keyword: {keyword}")
        urls = scrape_bing_image_urls(keyword, max_urls=120)

        for url in tqdm(urls):
            if count >= TARGET_COUNT:
                break

            filename = f"{dataset_name}_{count + 1:03d}.jpg"
            save_path = os.path.join(folder, filename)

            success = download_image(url, save_path)

            if success:
                count += 1

            time.sleep(0.25)

        print(f"{dataset_name}: now {count} images")

    print(f"Finished {dataset_name}: {count} images")


===== facades: currently 53 images =====

Scraping keyword: building facade architecture
building facade architecture: collected 25 urls
building facade architecture: collected 60 urls
building facade architecture: collected 95 urls
building facade architecture: collected 120 urls


100%|████████████████████████████████████████████████████████████████████████████████| 120/120 [01:45<00:00,  1.14it/s]


facades: now 173 images

Scraping keyword: modern building facade
modern building facade: collected 60 urls
modern building facade: collected 95 urls
modern building facade: collected 120 urls


 22%|██████████████████▏                                                              | 27/120 [00:23<01:21,  1.14it/s]


facades: now 200 images
Finished facades: 200 images

===== streets: currently 52 images =====

Scraping keyword: urban street city architecture
urban street city architecture: collected 60 urls
urban street city architecture: collected 95 urls
urban street city architecture: collected 120 urls


...

streets: now 200 images
Finished streets: 200 images

===== textures: currently 53 images =====

Scraping keyword: brick wall texture
brick wall texture: collected 29 urls
brick wall texture: collected 99 urls
brick wall texture: collected 120 urls


100%|████████████████████████████████████████████████████████████████████████████████| 120/120 [01:41<00:00,  1.18it/s]


textures: now 173 images

Scraping keyword: concrete wall texture
concrete wall texture: collected 60 urls
concrete wall texture: collected 120 urls


 22%|██████████████████▏                                                              | 27/120 [00:22<01:17,  1.20it/s]

textures: now 200 images
Finished textures: 200 images


In [11]:
for folder in ["facades", "streets", "textures"]:
    path = os.path.join("01_datasets", folder)
    files = [
        f for f in os.listdir(path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ] if os.path.exists(path) else []

    print(folder, len(files))

facades 200
streets 200
textures 200
